# Phase 1 — M_pop (population model)
Train the no-individual population model, save to Drive, reload, then TEST it: how well does it predict held-out behavior on each task? **GPU runtime (L4).**

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
if not os.path.exists('/content/sro-minitaur-transfer'):
    !git clone https://github.com/YifeiCAO/sro-minitaur-transfer.git
!cd /content/sro-minitaur-transfer && git pull
%cd /content/sro-minitaur-transfer
!pip -q install -e . && pip -q install -r requirements-model.txt
from huggingface_hub import login; login()  # paste HF token

### 1) Train M_pop  → saves LoRA adapter to Drive (~2-3h).  SKIP if already trained.

In [ ]:
!cd /content/sro-minitaur-transfer && python scripts/finetune_mpop.py \
    --subset starting_subset --max-seq-len 2048 \
    --out /content/drive/MyDrive/sro_minitaur/mpop

### 2) Read back — verify M_pop is on Drive

In [ ]:
!ls -la /content/drive/MyDrive/sro_minitaur/mpop   # expect adapter_config.json + adapter_model.safetensors

### 3) TEST — M_pop's per-task prediction on the held-out test set
For each task: response accuracy (model argmax == human's actual response) + NLL, vs a majority-response baseline. (~30-40 min for 11 tasks; add `--limit 15` for a quick look.)

In [ ]:
!cd /content/sro-minitaur-transfer && python scripts/phase1_test.py \
    --mpop /content/drive/MyDrive/sro_minitaur/mpop --subset starting_subset

### 4) Read back — per-task accuracy table

In [ ]:
import json, pandas as pd
r = json.load(open('/content/drive/MyDrive/sro_minitaur/phase1_test.json'))
print('macro accuracy:', r['macro_accuracy'])
pd.DataFrame(r['per_task'])